In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sqlite3

pd.options.display.max_columns = 999
pd.options.display.max_rows = 999

pd.set_option('max_colwidth', None)

In [2]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

In [3]:
def df_inj1020_fix_Date_Team(df_):
    df_.loc[5923, ['Team']] = 'Pacers'
    return df_

def get_5bsa1018_inj1018_player_diff():
    print(len(set(df_inj1018r.player) - set(df_5bsa1018.PLAYER_NAME)))
    print(set(df_inj1018r.player) - set(df_5bsa1018.PLAYER_NAME))
    print()
    
    print(len(set(df_5bsa1018.PLAYER_NAME) - set(df_inj1018r.player)))
    print(set(df_5bsa1018.PLAYER_NAME) - set(df_inj1018r.player))
    return 0
    

#return first 5 and last 5 rows of dataframe
def p(df_):
    return pd.concat([df_.head(), df_.tail()])



In [4]:
df_inj1018r = pd.read_csv(os.path.join('..', 'processed_data', '9game_date_team_relinquished_player_2010_2018.csv'))
print(df_inj1018r.shape)
p(df_inj1018r)

(14448, 5)


,game_date,Team,Relinquished,Notes,player
0,2010-10-26,Blazers,Elliot Williams,placed on IL,Elliot Williams
1,2010-10-26,Blazers,Greg Oden,placed on IL with left knee injury (out for season),Greg Oden
2,2010-10-26,Blazers,Joel Przybilla,placed on IL placed on IL recovering from surgery on right knee to repair ruptured patella tendon,Joel Przybilla
3,2010-10-26,Celtics,Avery Bradley,placed on IL recovering from surgery on left ankle,Avery Bradley
4,2010-10-26,Celtics,Kendrick Perkins,placed on IL recovering from surgery on right knee to repair torn ACL,Kendrick Perkins
14443,2018-05-09,Thunder,Paul George,arthroscopic surgery on right knee (out indefinitely),Paul George
14444,2018-05-08,Jazz,Dante Exum,placed on IL with strained left hamstring (out for season),Dante Exum
14445,2018-05-25,Rockets,Chris Paul,strained right hamstring (out for season),Chris Paul
14446,2018-05-22,Warriors,Andre Iguodala,bruised left leg (DTD),Andre Iguodala
14447,2018-05-26,Cavaliers,Kevin Love,concussion (DTD),Kevin Love


In [5]:
def get_injury_and_win_lose():
    #get injury report for 2010-11 thru 2017-18 sesasons (for player relinquished): df_inj1018r
    df_inj1018r = pd.read_csv(os.path.join('..', 'processed_data', '9game_date_team_relinquished_player_2010_2018.csv'), parse_dates=['game_date'])

    
    #read in nba boxscore advanced data for seasons 2009-10 thru 2017-18: df_5bsa1018
    df_3gdgiti0618 = pd.read_csv(os.path.join('..', 'processed_data', '3game_id_game_date_team_id.csv'), parse_dates=['game_date'])
    df_3gdgi0618 = df_3gdgiti0618[['game_date', 'game_id']].drop_duplicates()

    df_5bsa1018 = pd.read_csv(os.path.join('..', 'processed_data', '5boxscoreadvancedv2_2009_2018.csv'))
    df_5bsa1018.PLAYER_NAME = df_5bsa1018.PLAYER_NAME.fillna('')
    df_5bsa1018 = pd.merge(df_3gdgi0618, df_5bsa1018.rename(columns={'GAME_ID':'game_id'}), on='game_id', how='inner')
    df_5bsa1018.MIN = df_5bsa1018.MIN.fillna('00:00')
    df_5bsa1018.MIN = pd.to_timedelta('00:' + df_5bsa1018.MIN).astype('timedelta64[s]').astype('float64') / 60
    df_5bsa1018.TEAM_ID = df_5bsa1018.TEAM_ID.fillna(0).astype('int64')
    
    df_gdinj1018 = pd.merge(df_inj1018r, df_5bsa1018.rename(columns={'PLAYER_NAME':'player'}), how='inner', on=['game_date', 'player'])
    
    df_gdinj1018[['E_OFF_RATING', 'OFF_RATING',
       'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT',
       'AST_TOV', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT',
       'EFG_PCT', 'TS_PCT', 'USG_PCT', 'E_USG_PCT', 'E_PACE', 'PACE',
       'PACE_PER40', 'POSS', 'PIE']] = df_gdinj1018[['E_OFF_RATING', 'OFF_RATING',
       'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT',
       'AST_TOV', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT',
       'EFG_PCT', 'TS_PCT', 'USG_PCT', 'E_USG_PCT', 'E_PACE', 'PACE',
       'PACE_PER40', 'POSS', 'PIE']].fillna(0)
    
    df_gdinj1018['ig_player_injury'] = 0
    df_gdinj1018.loc[df_gdinj1018.MIN != 0, 'ig_player_injury'] = 1
    
    df_temp = df_gdinj1018[df_gdinj1018.MIN != 0][['game_id','TEAM_ID', 'ig_player_injury']]
    df_temp['pts'] = 0
    
    df_temp2 = df_3gdgiti0618[['game_id', 'team_id', 'pts']]
    df_temp2['ig_player_injury'] = 0
    df_temp2 = pd.merge(df_temp.game_id.drop_duplicates(), df_temp2, how='left', on='game_id')
    
    df_temp = pd.concat([df_temp.rename(columns={'TEAM_ID':'team_id'}), df_temp2])
    
    df_temp = df_temp.groupby(['game_id', 'team_id']).sum()
    
    df_temp.reset_index(inplace=True)
    
    df_temp = df_temp[(df_temp.game_id != 21000342) &
                                          (df_temp.game_id != 21000646) &
                                          (df_temp.game_id != 21500064) &
                                          (df_temp.game_id != 21600060) &
                                         (df_temp.game_id != 21600176) &
                                         (df_temp.game_id != 21600663) &
                                         (df_temp.game_id != 21600834) &
                                         (df_temp.game_id != 41500213) &
                                         (df_temp.game_id != 41600311)]
    
    game_id_winlose = set(df_temp.game_id)
    
    injury_wl = pd.DataFrame({'injury_&_win':[0], 'injury_&_lose':[0]})
    
    for gi in game_id_winlose:

        injury_wl['injury_&_win']

        df_temp[['ig_player_injury', 'pts']].reset_index(drop=True)

        df_ip = df_temp[df_temp.game_id == gi][['ig_player_injury', 'pts']].reset_index(drop=True)
        df_ip = df_ip.reset_index(drop=True)
        if (df_ip.iloc[0,0] > df_ip.iloc[1,0]) & (df_ip.iloc[0,1] > df_ip.iloc[1,1]):
            injury_wl['injury_&_win'] = injury_wl['injury_&_win'] + 1
        if (df_ip.iloc[0,0] > df_ip.iloc[1,0]) & (df_ip.iloc[0,1] < df_ip.iloc[1,1]):
            injury_wl['injury_&_lose'] = injury_wl['injury_&_lose'] + 1
        if (df_ip.iloc[0,0] < df_ip.iloc[1,0]) & (df_ip.iloc[0,1] < df_ip.iloc[1,1]):
            injury_wl['injury_&_win'] = injury_wl['injury_&_win'] + 1
        if (df_ip.iloc[0,0] < df_ip.iloc[1,0]) & (df_ip.iloc[0,1] > df_ip.iloc[1,1]):
            injury_wl['injury_&_lose'] = injury_wl['injury_&_lose'] + 1

    return injury_wl
    

In [6]:
def get_df_ts_check():
    #get injury report for 2010-11 thru 2017-18 sesasons (for player relinquished): df_inj1018r
    df_inj1018r = pd.read_csv(os.path.join('..', 'processed_data', '9game_date_team_relinquished_player_2010_2018.csv'), parse_dates=['game_date'])

    
    #read in nba boxscore advanced data for seasons 2009-10 thru 2017-18: df_5bsa1018
    df_3gdgiti0618 = pd.read_csv(os.path.join('..', 'processed_data', '3game_id_game_date_team_id.csv'), parse_dates=['game_date'])
    df_3gdgi0618 = df_3gdgiti0618[['game_date', 'game_id']].drop_duplicates()

    df_5bsa1018 = pd.read_csv(os.path.join('..', 'processed_data', '5boxscoreadvancedv2_2009_2018.csv'))
    df_5bsa1018.PLAYER_NAME = df_5bsa1018.PLAYER_NAME.fillna('')
    df_5bsa1018 = pd.merge(df_3gdgi0618, df_5bsa1018.rename(columns={'GAME_ID':'game_id'}), on='game_id', how='inner')
    df_5bsa1018.MIN = df_5bsa1018.MIN.fillna('00:00')
    df_5bsa1018.MIN = pd.to_timedelta('00:' + df_5bsa1018.MIN).astype('timedelta64[s]').astype('float64') / 60
    df_5bsa1018.TEAM_ID = df_5bsa1018.TEAM_ID.fillna(0).astype('int64')
    
    df_gdinj1018 = pd.merge(df_inj1018r, df_5bsa1018.rename(columns={'PLAYER_NAME':'player'}), how='inner', on=['game_date', 'player'])
    
    df_gdinj1018[['E_OFF_RATING', 'OFF_RATING',
       'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT',
       'AST_TOV', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT',
       'EFG_PCT', 'TS_PCT', 'USG_PCT', 'E_USG_PCT', 'E_PACE', 'PACE',
       'PACE_PER40', 'POSS', 'PIE']] = df_gdinj1018[['E_OFF_RATING', 'OFF_RATING',
       'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT',
       'AST_TOV', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT',
       'EFG_PCT', 'TS_PCT', 'USG_PCT', 'E_USG_PCT', 'E_PACE', 'PACE',
       'PACE_PER40', 'POSS', 'PIE']].fillna(0)
    
    df_gdinj1018['ig_player_injury'] = 0
    df_gdinj1018.loc[df_gdinj1018.MIN != 0, 'ig_player_injury'] = 1
    
    
    df_3gdgiti0618 = pd.read_csv(os.path.join('..', 'processed_data', '3game_id_game_date_team_id.csv'), parse_dates=['game_date'])
    
    df_ts_check = pd.merge(df_3gdgiti0618[['game_id', 'season']].drop_duplicates().reset_index(drop=True), df_gdinj1018, on=['game_id'], how='inner')
    
    return df_ts_check

In [7]:
get_injury_and_win_lose()

<ipython-input-5-28fd12b48048>:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp2['ig_player_injury'] = 0


,injury_&_win,injury_&_lose
0,87,102


In [8]:
os.getcwd()

'/Users/pauljacob/Documents/GitHub/DSCapstone2/02_Data_Wrangling'

In [9]:
df_inj1018r = pd.read_csv(os.path.join('..', 'processed_data', '9game_date_team_relinquished_player_2010_2018.csv'), parse_dates=['game_date'])
print(df_inj1018r.shape)
p(df_inj1018r)

(14448, 5)


,game_date,Team,Relinquished,Notes,player
0,2010-10-26,Blazers,Elliot Williams,placed on IL,Elliot Williams
1,2010-10-26,Blazers,Greg Oden,placed on IL with left knee injury (out for season),Greg Oden
2,2010-10-26,Blazers,Joel Przybilla,placed on IL placed on IL recovering from surgery on right knee to repair ruptured patella tendon,Joel Przybilla
3,2010-10-26,Celtics,Avery Bradley,placed on IL recovering from surgery on left ankle,Avery Bradley
4,2010-10-26,Celtics,Kendrick Perkins,placed on IL recovering from surgery on right knee to repair torn ACL,Kendrick Perkins
14443,2018-05-09,Thunder,Paul George,arthroscopic surgery on right knee (out indefinitely),Paul George
14444,2018-05-08,Jazz,Dante Exum,placed on IL with strained left hamstring (out for season),Dante Exum
14445,2018-05-25,Rockets,Chris Paul,strained right hamstring (out for season),Chris Paul
14446,2018-05-22,Warriors,Andre Iguodala,bruised left leg (DTD),Andre Iguodala
14447,2018-05-26,Cavaliers,Kevin Love,concussion (DTD),Kevin Love


In [10]:
#AAA. On the 2011-01-22 nba game Wade was NWT due to a migraine (by the nba.com boxscore) during this Heat vs. Raptors
# game. Was Wade inactive for this game according to the inactive player game-by-game dataframe df_ina1018?


path = os.path.join('..', 'raw_data','basketball.sqlite')
path
con = sqlite3.connect(path)
cursor = con.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())


[('Player',), ('Team',), ('Team_Attributes',), ('Team_History',), ('Player_Attributes',), ('Game_Officials',), ('Game_Inactive_Players',), ('Team_Salary',), ('Player_Salary',), ('Draft',), ('Draft_Combine',), ('Player_Photos',), ('Player_Bios',), ('Game',), ('News',), ('News_Missing',)]


In [11]:
df_ina = pd.read_sql_query("SELECT * from Game_Inactive_Players", con)
df_ina['player'] = df_ina.FIRST_NAME + " " + df_ina.LAST_NAME
df_ina.player = df_ina.player.str.strip()
df_ina

,PLAYER_ID,FIRST_NAME,LAST_NAME,JERSEY_NUM,TEAM_ID,TEAM_CITY,TEAM_NAME,TEAM_ABBREVIATION,GAME_ID,player
0,184,Bobby,Phills,14,1610612739,Cleveland,Cavaliers,CLE,0029600034,Bobby Phills
1,781,Will,Perdue,41,1610612759,San Antonio,Spurs,SAS,0029600034,Will Perdue
2,120,Steven,Smith,8,1610612737,Atlanta,Hawks,ATL,0029600132,Steven Smith
3,698,Eddie,Johnson,8,1610612754,Indiana,Pacers,IND,0029600203,Eddie Johnson
4,32,Clifford,Rozier,44,1610612753,Orlando,Magic,ORL,0029600213,Clifford Rozier
...,...,...,...,...,...,...,...,...,...,...
97932,2544,LeBron,James,23,1610612747,Los Angeles,Lakers,LAL,0022000982,LeBron James
97933,203115,Will,Barton,5,1610612743,Denver,Nuggets,DEN,0022000982,Will Barton
97934,1627750,Jamal,Murray,27,1610612743,Denver,Nuggets,DEN,0022000982,Jamal Murray
97935,1630192,Zeke,Nnaji,22,1610612743,Denver,Nuggets,DEN,0022000982,Zeke Nnaji


In [12]:
df_ina.GAME_ID = df_ina.GAME_ID.astype(int)
df_ina.GAME_ID.head()

0    29600034
1    29600034
2    29600132
3    29600203
4    29600213
Name: GAME_ID, dtype: int64

In [13]:
df_ina.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97937 entries, 0 to 97936
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   PLAYER_ID          97937 non-null  object
 1   FIRST_NAME         97937 non-null  object
 2   LAST_NAME          97937 non-null  object
 3   JERSEY_NUM         97937 non-null  object
 4   TEAM_ID            97937 non-null  object
 5   TEAM_CITY          97937 non-null  object
 6   TEAM_NAME          97937 non-null  object
 7   TEAM_ABBREVIATION  97937 non-null  object
 8   GAME_ID            97937 non-null  int64 
 9   player             97937 non-null  object
dtypes: int64(1), object(9)
memory usage: 7.5+ MB


In [14]:
#stick a date for game_id: get (game_id, game_date, team_id) dataframe df_3gigdti1018
path = os.path.join('..', 'processed_data', '3game_id_game_date_team_id.csv')
path
df_3gigdti = pd.read_csv(path)
df_3gigdti1018 = df_3gigdti[(df_3gigdti.season == '2010-11') |
                            (df_3gigdti.season == '2011-12') |
                            (df_3gigdti.season == '2012-13') |
                            (df_3gigdti.season == '2013-14') |
                            (df_3gigdti.season == '2014-15') |
                            (df_3gigdti.season == '2015-16') |
                            (df_3gigdti.season == '2016-17') |
                            (df_3gigdti.season == '2017-18')]
df_3gigdti1018.head()

,game_id,game_date,team_id,is_home,w_pct,min,fgm,fga,fg_pct,fg3m,fg3a,fg3_pct,ftm,fta,ft_pct,oreb,dreb,reb,ast,stl,blk,tov,pf,pts,season_year,season_type,season
9840,21000001,2010-10-26,1610612738,t,1.0,240,32.0,69.0,0.464,8.0,16.0,0.500,16.0,25.0,1.0,8.0,34.0,42,25.0,6.0,4.0,18.0,19.0,88,2010,Regular Season,2010-11
9841,21000001,2010-10-26,1610612748,f,0.0,240,27.0,74.0,0.365,8.0,20.0,0.400,18.0,25.0,1.0,11.0,28.0,39,15.0,10.0,6.0,17.0,21.0,80,2010,Regular Season,2010-11
9842,21000002,2010-10-26,1610612757,t,1.0,240,43.0,93.0,0.462,10.0,20.0,0.500,10.0,15.0,1.0,18.0,30.0,48,31.0,11.0,2.0,12.0,22.0,106,2010,Regular Season,2010-11
9843,21000002,2010-10-26,1610612756,f,0.0,240,36.0,74.0,0.486,9.0,19.0,0.474,11.0,16.0,1.0,7.0,23.0,30,15.0,3.0,4.0,19.0,19.0,92,2010,Regular Season,2010-11
9844,21000003,2010-10-26,1610612745,f,0.0,240,38.0,91.0,0.418,8.0,20.0,0.400,26.0,28.0,1.0,16.0,37.0,53,25.0,6.0,7.0,20.0,25.0,110,2010,Regular Season,2010-11


In [15]:
df_3gigdti1018[['game_id', 'game_date', 'team_id']].info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 20534 entries, 9840 to 31037
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   game_id    20534 non-null  int64 
 1   game_date  20534 non-null  object
 2   team_id    20534 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 641.7+ KB


In [16]:
df_ina.rename(columns={'GAME_ID':'game_id'}).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97937 entries, 0 to 97936
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   PLAYER_ID          97937 non-null  object
 1   FIRST_NAME         97937 non-null  object
 2   LAST_NAME          97937 non-null  object
 3   JERSEY_NUM         97937 non-null  object
 4   TEAM_ID            97937 non-null  object
 5   TEAM_CITY          97937 non-null  object
 6   TEAM_NAME          97937 non-null  object
 7   TEAM_ABBREVIATION  97937 non-null  object
 8   game_id            97937 non-null  int64 
 9   player             97937 non-null  object
dtypes: int64(1), object(9)
memory usage: 7.5+ MB


In [17]:
#stick a date for game_id: merge
df_ina1018 = pd.merge(df_3gigdti1018[['game_id', 'game_date']], df_ina.rename(columns={'GAME_ID':'game_id'}), on='game_id', how='inner').drop_duplicates()
df_ina1018

,game_id,game_date,PLAYER_ID,FIRST_NAME,LAST_NAME,JERSEY_NUM,TEAM_ID,TEAM_CITY,TEAM_NAME,TEAM_ABBREVIATION,player
0,21000001,2010-10-26,202376,Luke,Harangody,55,1610612738,Boston,Celtics,BOS,Luke Harangody
1,21000001,2010-10-26,2570,Kendrick,Perkins,43,1610612738,Boston,Celtics,BOS,Kendrick Perkins
2,21000001,2010-10-26,202340,Avery,Bradley,0,1610612738,Boston,Celtics,BOS,Avery Bradley
3,21000001,2010-10-26,202354,Dexter,Pittman,45,1610612748,Miami,Heat,MIA,Dexter Pittman
4,21000001,2010-10-26,2034,Mike,Miller,13,1610612748,Miami,Heat,MIA,Mike Miller
...,...,...,...,...,...,...,...,...,...,...,...
86249,21701230,2018-04-11,202697,Iman,Shumpert,4,1610612758,Sacramento,Kings,SAC,Iman Shumpert
86250,21701230,2018-04-11,203900,Markel,Brown,26,1610612745,Houston,Rockets,HOU,Markel Brown
86251,21701230,2018-04-11,201583,Ryan,Anderson,33,1610612745,Houston,Rockets,HOU,Ryan Anderson
86252,21701230,2018-04-11,2772,Trevor,Ariza,1,1610612745,Houston,Rockets,HOU,Trevor Ariza


NameError: name 'df_5bsa1018' is not defined

In [ ]:
Adf_inj1018 = df_inj1018r
print(Adf_inj1018.shape)
p(Adf_inj1018)

In [ ]:
#get game_date, season pairs
df_3gds = pd.read_csv(os.path.join('..', 'processed_data', '3game_id_game_date_team_id.csv'), parse_dates=['game_date'])[['game_date', 'season']].drop_duplicates()
df_3gds.shape



In [ ]:
#add season to Adf_inj1018
Adf_inj1018 = pd.merge(df_3gds, Adf_inj1018, on='game_date', how='inner')
Adf_inj1018



In [ ]:
#grab df_ts_check code


In [ ]:
#
pd.merge(Adf_inj1018, df_5bsa1018)

In [ ]:
#add team_id to injury report of 2010-11 thru 2017-18 seasons

pd.read_csv(os.path.join('..', 'raw_data', 'teams.csv'))



In [ ]:
#pd.read_csv(os.path.join('..', 'processed_data', '8franchise_start_end.csv'))

In [ ]:
Adf_5bsa1018 = df_5bsa1018[df_5bsa1018.MIN == 0]
print(Adf_5bsa1018.shape)
Adf_5bsa1018.head()

In [ ]:
get_df_ts_check()

In [ ]:
df_ts_check = pd.read_csv(os.path.join('..', 'processed_data', '10season_team_team_id_2010_2018.csv'))
print(df_ts_check.shape)
df_ts_check

In [ ]:
print(Adf_inj1018.shape)
p(Adf_inj1018)
Adf_inj1018m = pd.merge(Adf_inj1018, df_ts_check, on=['Team', 'season'], how='outer', indicator=True)
Adf_inj1018m

<font color='red'>hornet game_date range for the 2012-13 season was??</font> -around mid april?, https://statsdmz.nba.com/pdfs/20130417/20130417_NOHDAL.pdf

In [ ]:
Adf_inj1018m._merge.value_counts()

In [ ]:
pd.merge(Adf_inj1018, Adf_5bsa1018.rename(columns={'PLAYER_NAME':'player'}), how='outer', on=['game_date', 'player'], indicator=True)

In [ ]:
pd.merge(Adf_inj1018, Adf_5bsa1018.rename(columns={'PLAYER_NAME':'player'}), how='outer', on=['game_date', 'player'], indicator=True)._merge.value_counts()

In [ ]:
pd.merge(Adf_inj1018, Adf_5bsa1018.rename(columns={'PLAYER_NAME':'player'}), how='inner', on=['game_date', 'player'])

In [ ]:
Adf_ina1018 = df_ina1018
Adf_ina1018

In [ ]:
#df.select_dtypes('object')

In [ ]:
#df[''].value_counts().head()


In [ ]:
#missing_x = df[[' ', ' ']].isnull().sum(axis=1)

In [ ]:
df.hist(figsize=(15,10))
plt.subplots_adjust(hspace=.5);

In [ ]:
#df.loc[5, ' '] = 1111

In [ ]:

df.column_name.hist(bins=30)
plt.xlabel('column_name')
plt.ylabel('Count')
plt.title('Distribution of');

In [ ]:
df.column_name[df.column_name < 1000].describe()

In [ ]:
df.iloc[115]

In [ ]:
#type()

In [ ]:
#len()

In [ ]:
#df.iloc[:, 4]

In [ ]:
#df.iloc[:, [0,5,6]].copy()

In [ ]:
#set(df.column_name) - set(df2.column_name)

In [ ]:
#df.state[df.column_name.str.contains('state1|state2|state3|state4')]

In [ ]:
#regex
#df.column_name.replace(to_replace='\[.*\]', value='', regex=True, inplace=True)

In [ ]:
#df.plot(x='column_name1', y='column_name2', kind='scatter');

In [ ]:
#df.loc[df.column_name == 'state1', ['column_name', 'column_name']]

In [ ]:
#missing = pd.concat([df.isnull().sum(axis=1), 100 * df.isnull().mean(axis=1)], axis=1)
#missing.columns=['count', '%']
#missing.sort_values(by='count', ascending=False).head(10)

In [ ]:
#missing['%'].unique()

In [ ]:
#datapath = 'data'

In [ ]:
#if not os.path.exists(datapath):
    #os.mkdir(datapath)

In [ ]:
#datapath_df = os.path.join(datapath, 'df_cleaned.csv')
#if not os.path.exists(datapath_df):
#    df.to_csv(datapath_df, index=False)

In [ ]:
#datapath_df2 = os.path.join(datapath, 'df2.csv')
#if not os.path.exists(datapath_df2):
#    df2.to_csv(datapath_df2, index=False)